### Incremental merge for the dimension tables

In [0]:
from pyspark.sql.functions import (col,upper,trim,concat_ws,coalesce,lit,udf,when,max,create_map,to_timestamp,date_sub,to_date,first,lower,size,split)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
tables=spark.catalog.listTables("f1_warehouse.gold")

dim={}

for table in tables:
    df=spark.table(f"f1_warehouse.gold.{table.name}")
    dim[table.name]=df

In [0]:
for table_names,df in dim.items():
    print(table_names)

In [0]:
tables=spark.catalog.listTables("f1_warehouse.silver")

silver={}

for table in tables:
    df=spark.table(f"f1_warehouse.silver.{table.name}")
    silver[table.name]=df

In [0]:
for table_names,df in silver.items():
    print(table_names)

### dim_teams

In [0]:
silver["openf1_drivers"].show(5)

In [0]:
dim["dim_teams"].show(5)

dim["dim_teams"].printSchema()

In [0]:
dim_teams=dim["dim_teams"]
of1_drivers=silver["openf1_drivers"]


In [0]:
incoming_teams = (
    of1_drivers
    .select("team_colour", "team_name")
    .filter(col("team_name").isNotNull()) #cos null survives anti null join
    .dropDuplicates()
)

In [0]:
#get teams that are not in dim_tems
new_teams=incoming_teams.join(dim_teams,[incoming_teams.team_name==dim_teams.name],"leftanti")


In [0]:
new_teams.show()

In [0]:
#get max constructor id

max_constructor_id=(dim_teams.select(max("constructor_id")).collect()[0][0])

#add new constructor id to new teams
window=Window.orderBy("team_name")

new_teams=(new_teams.withColumn("constructor_id",F.row_number().over(window)+max_constructor_id))

#rename team_name to name 

new_teams=new_teams.withColumnRenamed("team_name","name")



In [0]:
#keep new_teams schema to match dim_teams schema
#rename team_name to name 

new_teams=(new_teams.withColumn("nationality",F.lit(None).cast("string"))
           .withColumn("url",F.lit(None).cast("string"))
           .withColumn("team_name",upper(col("name")))
           .select(
               "team_name",
               "constructor_id",
               "name",
               "nationality",
               "team_colour",
               "url"
           ))
       

In [0]:
dim_teams.count()

In [0]:
# merge into delta dimension

#first acess delta table

delta_dim_teams = DeltaTable.forName(spark,"f1_warehouse.gold.dim_teams")

#merge new teams into dim_teams

(delta_dim_teams.alias("dim_teams").merge(
    new_teams.alias("new_teams"),
    "dim_teams.name=new_teams.name").whenNotMatchedInsert(
        values={
            "name":"new_teams.name",
            "constructor_id":"new_teams.constructor_id",
            "nationality":"new_teams.nationality",
            "team_colour":"new_teams.team_colour",
            "url":"new_teams.url"
        })
.execute())

In [0]:
print(spark.table("f1_warehouse.gold.dim_teams").count())

### dim_driver


In [0]:
dim["dim_driver"].filter(col("code").isin("HUL","PER")).show()

In [0]:
silver["openf1_drivers"].show(5)

In [0]:
dim_driver=dim["dim_driver"]
openf1_drivers=silver["openf1_drivers"]

In [0]:
incoming_drivers=(openf1_drivers.select("driver_number","first_name","last_name","name_acronym").filter(col("first_name").isNotNull() &
        col("last_name").isNotNull()).dropDuplicates())

new_drivers=(incoming_drivers.alias("o").join(dim_driver.alias("d"),
                                              (col("o.first_name")==col("d.forename"))&
                                              (col("o.last_name")==col("d.surname")),
                                              "leftanti")
            .select("o.*")
            .withColumnRenamed("first_name","forename")
            .withColumnRenamed("last_name","surname")
            .withColumnRenamed("name_acronym","code")
            .withColumnRenamed("driver_number","number"))

In [0]:
new_drivers.count()

In [0]:
#get max driver id

max_driver_id=dim_driver.selectExpr("max(driver_id)").collect()[0][0]


#add new constructor id to new drivers
window=Window.orderBy("number")

new_drivers=(new_drivers.withColumn("driver_id",F.row_number().over(window)+max_driver_id))




In [0]:
new_drivers.show(5)


In [0]:
new_drivers=(new_drivers
            .withColumn("is_current_driver",F.lit(1).cast("int"))
            .withColumn("dob",F.lit(None).cast("date"))
            .withColumn("url",F.lit(None).cast("string"))
            .withColumn("nationality",F.lit(None).cast("string"))
            .withColumn("driver_ref",lower(col("surname"))).select(
                "driver_id","driver_ref","number","code","forename","surname","dob","nationality","url","is_current_driver"
            ))

In [0]:
dim_driver.count()

In [0]:
#merge into delta

delta_dim_driver=DeltaTable.forName(spark,"f1_warehouse.gold.dim_driver")

#insert new_drivers
#delta merge is wrapped to brackets so python can use method chain across multiple lines 

(delta_dim_driver.alias("d").merge(
    new_drivers.alias("n"),
    "d.forename=n.forename AND d.surname=n.surname"
).whenNotMatchedInsert(values={
    "driver_id":"n.driver_id",
    "driver_ref":"n.driver_ref",
    "number":"n.number",
    "code":"n.code",
    "forename":"n.forename",
    "surname":"n.surname",
    "dob":"n.dob",
    "nationality":"n.nationality",
    "url":"n.url",
    "is_current_driver":"n.is_current_driver"
}).execute())

In [0]:
delta_dim_driver.toDF().count()

### dim_circuit

In [0]:
silver["openf1_meetings"].show(5)

In [0]:
dim["dim_circuits"].show(5)

In [0]:
dim_circuit = dim["dim_circuits"]
of1_circuits = silver["openf1_meetings"]

In [0]:
of1_circuits=of1_circuits.withColumn("location",
     when(col("location") == "Spa-Francorchamps", "Spa")
    .when(col("location") == "Lusail", "Al Daayen")
    .when(col("location") == "Bahrain", "Sakhir")
    .when(col("location").isin("Yas Island","Yas Marina"),"Abu Dhabi")
    .when(col("location") == "Miami Gardens", "Miami")
    .when(col("location").isin("Monte Carlo","Monaco"), "Monte-Carlo")
    .when(col("location") == "Montréal", "Montreal")
    .otherwise(col("location")) )

In [0]:

incoming_circuits = (
    of1_circuits
    .select("location","country_name","meeting_name","circuit_type","circuit_short_name")
    .dropDuplicates()
)


new_circuits = (
    incoming_circuits.alias("o")
    .join(
        dim_circuit.alias("d"),
        col("o.location") == col("d.location"),
        "leftanti"
    )
    .select("o.*")
)

In [0]:
new_circuits.show()

In [0]:
max_circuit_id = (
    dim_circuit
    .select(max("circuit_id"))
    .collect()[0][0]
)

window = Window.orderBy("location")

new_circuits = (
    new_circuits
    .withColumn(
        "circuit_id",
        F.row_number().over(window) + max_circuit_id
    )
)

In [0]:
new_circuits=(new_circuits
              .withColumnRenamed("country_name","country")
              .withColumnRenamed("circuit_short_name","name")
              .withColumn("circuit_ref",lit(None))
              .withColumn("lat",lit(0))
              .withColumn("lng",lit(0))
              .withColumn("alt",lit(0))
              .select("circuit_id","name","location","country","lat","lng","alt","circuit_ref","circuit_type"))
     
        
            

In [0]:
dim_circuit.count()

In [0]:
dim_circuit.printSchema()

In [0]:
new_circuits.printSchema()

In [0]:
delta_dim_circuit = DeltaTable.forName(
    spark,
    "f1_warehouse.gold.dim_circuits"
)

(
    delta_dim_circuit.alias("d")
    .merge(
        new_circuits.alias("s"),
        "d.location = s.location"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
delta_dim_circuit.toDF().count()

### dim_meetings

In [0]:
dim_meeting = dim["dim_meetings"]
of1_meetings= silver["openf1_meetings"]

In [0]:
of1_meetings.show(5)

In [0]:
dim_meeting.show(5)

In [0]:
of1_meetings=of1_meetings.withColumn("location",
     when(col("location") == "Spa-Francorchamps", "Spa")
    .when(col("location") == "Lusail", "Al Daayen")
    .when(col("location") == "Bahrain", "Sakhir")
    .when(col("location").isin("Yas Island","Yas Marina"),"Abu Dhabi")
    .when(col("location") == "Miami Gardens", "Miami")
    .when(col("location").isin("Monte Carlo","Monaco"), "Monte-Carlo")
    .when(col("location") == "Montréal", "Montreal")
    .otherwise(col("location")) )

In [0]:
incoming_meetings = (
    of1_meetings
    .select(
        "is_cancelled",
        "meeting_name",
        "date_end",
        "date_start",
        "meeting_id",
        "year",
        "location"
    )
    .withColumnRenamed("meeting_id", "openf1_meeting_id")
)

In [0]:
# create round column

window = Window.partitionBy("year").orderBy("date_start")

incoming_meetings = (
    incoming_meetings
    .withColumn(
        "round",
        F.when(
            F.col("meeting_name") == "Pre Season Testing",
            0
        ).otherwise(
            F.row_number().over(window) - 1
        )
    )
)

In [0]:
dim_circuiit=dim["dim_circuits"].select("location","circuit_id")
incoming_meetings=incoming_meetings.join(dim_circuiit,["location"],"left")


In [0]:
incoming_meetings.show()

In [0]:
#new meetings

new_meetings = (
    incoming_meetings.alias("n")
    .join(
        dim_meeting.alias("d"),
        (F.col("n.year") == F.col("d.year")) &
        (F.col("n.openf1_meeting_id") == F.col("d.openf1_meeting_id")) &
        (F.col("n.circuit_id") == F.col("d.circuit_id")),
        "leftanti"
    )
)

In [0]:
new_meetings.show(5)

In [0]:
max_meeting_id = (
    dim_meeting
    .select(max("race_meeting_id"))
    .collect()[0][0]
)

window = Window.orderBy("openf1_meeting_id")

new_meetings= (
    new_meetings
    .withColumn(
        "race_meeting_id",
        F.row_number().over(window) + max_meeting_id
    )
)

In [0]:
dim_meeting.printSchema()

In [0]:
new_meetings.printSchema()

In [0]:
new_meetings = (
    new_meetings
    .withColumn(
        "kaggle_race_id",
        F.lit(None).cast("int")
    )
    .withColumn(
        "date_start",
        F.to_date("date_start")
    )
    .withColumn(
        "date_end",
        F.to_date("date_end")
    )
    .select(
        "openf1_meeting_id",
        "kaggle_race_id",
        "circuit_id",
        "year",
        "round",
        "meeting_name",
        "is_cancelled",
        "date_start",
        "date_end",
        "race_meeting_id"
    )
)

In [0]:
dim_meeting.count()

In [0]:
delta_dim_meeting = DeltaTable.forName(
    spark,
    "f1_warehouse.gold.dim_meetings"
)

(
    delta_dim_meeting.alias("d")
    .merge(
        new_meetings.alias("n"),
        """
        d.year = n.year
        AND d.openf1_meeting_id = n.openf1_meeting_id
        AND d.circuit_id = n.circuit_id
        """
    )
    .whenNotMatchedInsert(
        values={
            "race_meeting_id": "n.race_meeting_id",
            "openf1_meeting_id": "n.openf1_meeting_id",
            "kaggle_race_id": "n.kaggle_race_id",
            "circuit_id": "n.circuit_id",
            "year": "n.year",
            "round": "n.round",
            "meeting_name": "n.meeting_name",
            "is_cancelled": "n.is_cancelled",
            "date_start": "n.date_start",
            "date_end": "n.date_end"
        }
    )
    .execute()
)

In [0]:
delta_dim_meeting.toDF().count()